<!-- course-header -->
# Lesson 18: Fine-tuning language models

*Worked solutions for lesson 18. Read the lesson [online](https://ashutoshjoshi1.github.io/AI-ZerotoMastery/lessons/18-fine-tuning.html) or [offline](../../lessons/18-fine-tuning.html). The exercise version without solutions is `../18-fine-tuning.ipynb`.*

<a href="https://colab.research.google.com/github/ashutoshjoshi1/AI-ZerotoMastery/blob/main/notebooks/solutions/18-fine-tuning-solutions.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

Learn when fine-tuning is the right tool and when it is not, understand LoRA and QLoRA down to the matrix shapes and memory budget, fine-tune a small open model on your own task and measure it against a prompted baseline, then see where preference tuning, distillation and adapter serving take you next.

**Runs on:** A GPU runtime (a free Colab T4 is enough for the 0.5B model) for the training cells. The LoRA-from-scratch code, the data generation, the memory calculator and the baseline exercises run on any CPU.

**How to use it:** read a section of the lesson, then run the matching cells here, in order. Each exercise has a cell titled *Hands-on* with the same name as in the lesson. Change numbers, break things on purpose, and note what happens in your learning log. Cells that need an API key, a GPU or a download say so in their first line.

## Setup

Install what this lesson needs. Skip this cell if your environment from lesson 00 already has these packages.

In [ ]:
%pip install -q torch transformers datasets peft trl accelerate bitsandbytes

## When to fine-tune, and when not to

Fine-tuning means continuing to train a pretrained model on your own examples so its weights change. Work through the cheaper options first: prompting (lesson 11), retrieval when the model lacks knowledge (lesson 14), and tools when it needs live data or exact computation (lesson 15). Fine-tuning earns its place when you need to change **behaviour** in ways prompts cannot reliably achieve, or need the same behaviour **cheaper or faster**.

> **Before you fine-tune, have an eval set**
>
> Without the evaluation discipline of lesson 17 you cannot tell whether fine-tuning helped, hurt, or just changed things. Measure the prompted baseline first. Fine-tuning that does not beat a well-prompted baseline on your eval set is wasted money.

### Full fine-tuning versus parameter-efficient fine-tuning

**Full fine-tuning** updates every weight. With the Adam optimiser in mixed precision, a common rule of thumb is about 16 bytes of GPU memory per parameter (16-bit weights and gradients, a 32-bit master copy, and two 32-bit optimiser moments), before activations. For a 7-billion-parameter model that is over 110 GB.

**Parameter-efficient fine-tuning (PEFT)** freezes the pretrained weights and trains a small number of new parameters. Memory needs drop dramatically, and the result is a small file you can swap in and out. **LoRA** is the method nearly everyone uses.

### Hands-on: the memory budget of fine-tuning

1. Write `finetune_memory_gb(params_b, method)` that returns (weights, training state) in GB from the rules in this section: full fine-tuning costs 16 bytes per parameter in total; LoRA keeps a frozen 16-bit base and trains about 0.6% of the parameters, each with its own 16 bytes of state; QLoRA stores the base in 4-bit, about 0.55 bytes per parameter once the quantisation scales are counted.
2. Print a table for 0.5B, 7B and 70B models and the three methods.
3. For each row, say which single GPU it fits on before activations (16, 24, 48 or 80 GB), or "multi-GPU".

You are done when the 7B row matches the table in "A memory budget, worked out" below (about 112 GB full, 15 GB LoRA, 5 GB QLoRA) and you can say why a 70B model still needs a large card even with QLoRA.

#### Solution: a solution

In [ ]:
def finetune_memory_gb(params_b, method, lora_fraction=0.006):
    """Rough GPU memory in GB for the weights and the training state, before activations."""
    n = params_b * 1e9
    if method == "full":        # 2 (weights) + 2 (grads) + 4 (master) + 4 + 4 (Adam moments) bytes per param
        weights, state = 2 * n, 14 * n
    elif method == "lora":      # frozen 16-bit base; only the adapter carries gradients and optimiser state
        weights, state = 2 * n, 16 * lora_fraction * n
    elif method == "qlora":     # frozen 4-bit base (about 0.55 bytes per param including the scales)
        weights, state = 0.55 * n, 16 * lora_fraction * n
    return weights / 1e9, state / 1e9

def fits_on(gb):
    return next((f"{g} GB GPU" for g in [16, 24, 48, 80] if gb <= 0.85 * g), "multi-GPU")

print(f"{'model':>6} {'method':<6} {'weights':>8} {'state':>7} {'total':>7}   fits on")
for p in [0.5, 7, 70]:
    for m in ["full", "lora", "qlora"]:
        w, s = finetune_memory_gb(p, m)
        print(f"{p:>5}B {m:<6} {w:>8.1f} {s:>7.1f} {w + s:>7.1f}   {fits_on(w + s)}")

The 7B row gives 112 GB, 14.7 GB and 4.5 GB. Even QLoRA of a 70B model needs about 45 GB before activations, because 4-bit weights of 70 billion parameters are still 38 GB; that is why 70B fine-tunes live on 80 GB cards. The 15% headroom in `fits_on` stands in for activations, which the table in "A memory budget, worked out" treats separately: they scale with batch size and sequence length, not with the method.

## LoRA: low-rank adaptation

### The idea

Fine-tuning changes a weight matrix $W$ (shape $d_{\text{out}} \times d_{\text{in}}$) by some update $\Delta W$. These updates have low **intrinsic rank**, so LoRA learns them as the product of two thin matrices:

$$\Delta W = B A, \qquad B \in \mathbb{R}^{d_{\text{out}} \times r},\quad A \in \mathbb{R}^{r \times d_{\text{in}}},\quad r \ll d$$

and computes the layer as $h = W x + \frac{\alpha}{r} B A x$ with $W$ frozen. $B$ starts at zero, so training begins from exactly the original model. For a 4,096 × 4,096 projection, LoRA with $r = 16$ trains 131,072 parameters instead of 16.8 million.

### LoRA from scratch

In [ ]:
# Needs PyTorch; a GPU runtime is much faster
import torch, torch.nn as nn

class LoRALinear(nn.Module):
    """Wrap a frozen nn.Linear with a trainable low-rank update."""
    def __init__(self, base: nn.Linear, r: int = 8, alpha: int = 16):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False                                       # freeze W and b
        self.A = nn.Parameter(torch.randn(r, base.in_features) * 0.01)    # (r, d_in)
        self.B = nn.Parameter(torch.zeros(base.out_features, r))          # (d_out, r): zero, so no change at start
        self.scale = alpha / r

    def forward(self, x):                                                 # x: (..., d_in)
        return self.base(x) + (x @ self.A.T @ self.B.T) * self.scale     # (..., d_out)

    @torch.no_grad()
    def merge(self) -> nn.Linear:
        """Fold the update into W for zero-overhead inference."""
        self.base.weight += (self.B @ self.A) * self.scale
        return self.base

layer = LoRALinear(nn.Linear(4096, 4096), r=16)
trainable = sum(p.numel() for p in layer.parameters() if p.requires_grad)
total = sum(p.numel() for p in layer.parameters())
print(trainable, total, f"{100 * trainable / total:.2f}%")   # 131072 16912384 0.78%

Because $BA$ can be **merged** into $W$ after training, a LoRA-tuned model runs exactly as fast as the original. Or keep adapters separate and switch between them per request.

### Hyperparameters that matter

- **Rank $r$**: 8 to 64 is typical. Higher ranks can learn more complex changes but need more data.
- **Alpha**: often set to $r$ or $2r$.
- **Target modules**: applying LoRA to **all linear layers** (including the MLP) usually works better than attention projections only.
- **Learning rate**: much higher than for full fine-tuning, often around $10^{-4}$ to $2\times10^{-4}$.
- **Epochs**: 1 to 3. More risks memorising the training set.

### Hands-on: find the rank a task needs

1. Create a frozen base layer `nn.Linear(64, 64)` and a hidden "ideal" update $\Delta W$ of rank 4 (the product of two random 64 × 4 and 4 × 64 matrices, scaled down). The target is $y = (W + \Delta W)\,x$ on 2,048 random inputs.
2. For $r$ in 1, 2, 4, 8 and 16, wrap a copy of the same base layer in `LoRALinear` and train only $A$ and $B$ with Adam for 300 steps on the mean squared error to the target. Print the final loss per rank.
3. For the last model, compare its outputs before and after `merge()`.

You are done when the loss drops by orders of magnitude once $r$ reaches the true rank of 4 and barely improves beyond it, and the merged layer reproduces the unmerged outputs to within $10^{-5}$.

#### Solution: a solution

In [ ]:
torch.manual_seed(0)
d = 64
base = nn.Linear(d, d)
true_delta = (torch.randn(d, 4) @ torch.randn(4, d)) / d        # a hidden rank-4 change to W
X = torch.randn(2048, d)
with torch.no_grad():
    Y = base(X) + X @ true_delta.T                               # what the "fine-tuned" layer should output

for r in [1, 2, 4, 8, 16]:
    torch.manual_seed(r)
    layer = LoRALinear(nn.Linear(d, d), r=r, alpha=2 * r)
    layer.base.load_state_dict(base.state_dict())                # the same frozen base every time
    opt = torch.optim.Adam([layer.A, layer.B], lr=1e-2)
    for step in range(300):
        opt.zero_grad()
        loss = ((layer(X) - Y) ** 2).mean()
        loss.backward(); opt.step()
    print(f"r = {r:<2} final loss {loss.item():.2e}")

with torch.no_grad():
    before = layer(X)
    after = layer.merge()(X)                                     # merge folds BA into W; the output must not change
print("merged matches unmerged:", (before - after).abs().max().item() < 1e-5)

Ranks 1 and 2 plateau because a rank-2 product cannot represent a rank-4 matrix, whatever the optimiser does. Rank 4 drops the loss by orders of magnitude; 8 and 16 fit no better, they only train more parameters. Real fine-tuning updates are not exactly low rank, which is why 8 to 64 is the usual range and why rank is a hyperparameter you tune on validation loss rather than derive. The merge check is the property that lets LoRA models serve at full speed.

### QLoRA: fine-tuning on one consumer GPU

**QLoRA** stores the frozen base model in **4-bit** precision (NF4) and trains LoRA adapters in 16-bit on top. A 7B model's weights take roughly 4 to 5 GB instead of 14. With paged optimisers and gradient checkpointing, fine-tuning a 7B model fits on a single 16 to 24 GB GPU, and small models (0.5B to 3B) fit on a free Colab T4. Lesson 20 covers the quantisation formats themselves.

### A memory budget, worked out

| What | Full fine-tune, 7B | LoRA, 7B base in 16-bit | QLoRA, 7B base in 4-bit |
|---|---|---|---|
| Base weights | 14 GB | 14 GB | about 4 GB |
| Gradients + optimiser + master weights | about 98 GB | under 1 GB (40M params) | under 1 GB |
| Activations (batch-dependent) | several GB | several GB | several GB (less with checkpointing) |
| **Fits on** | multiple 80 GB GPUs | one 24 to 40 GB GPU | one 16 to 24 GB GPU |

Lesson 29 works out the activations row properly, as a function of batch and sequence length.

## Hands-on: fine-tune a small model

### The task

Teach a small open model to turn free-text field-technician notes into a strict JSON record. This is a textbook fine-tuning case: narrow, high-volume, format-sensitive, and something a 0.5B to 1.5B model can learn well enough to run cheaply or on a laptop.

### The data

Training data for chat models is a list of conversations in the same chat format the model was instruction-tuned with:

*train.jsonl (one example per line, shown pretty-printed)*

```json
{"messages": [
  {"role": "system", "content": "Convert the technician note to JSON with keys instrument_id, action, parts, duration_min, follow_up."},
  {"role": "user", "content": "P-117 coastal. swapped desiccant + new belt on filter wheel, 45 min. check again in 2 wks"},
  {"role": "assistant", "content": "{\"instrument_id\": \"P-117\", \"action\": \"maintenance\", \"parts\": [\"desiccant\", \"filter wheel belt\"], \"duration_min\": 45, \"follow_up\": \"recheck in 2 weeks\"}"}
]}
```

**Where do hundreds of examples come from?** Real historical records, labelled by hand, are best. A common shortcut is **distillation**: a large model labels a few hundred realistic inputs, you review a sample, then you fine-tune the small model on the result.

- **Quality beats quantity.** A few hundred clean, consistent, diverse examples often beat thousands of noisy ones.
- **Cover the real distribution**: typos, abbreviations, missing fields, multiple instruments in one note, notes that should produce `null`s.
- **Hold out** 10 to 20% as a validation set, and keep your separate eval set untouched.

### Hands-on: build the training set and a baseline to beat

The next cell is a small generator that writes synthetic technician notes and their JSON records, so the training cells run without any API. Run it even if you skip the exercise: the training cells read its files.

1. Run the generator to write `train.jsonl` (400 examples) and `val.jsonl` (60). Read ten examples. What variation does it cover, and what real-world variation is missing?
2. Write `grade(pred, gold)` that returns JSON validity and an exact-match flag per field.
3. Write a rule-based baseline `rule_convert(note)`: a regex for the instrument id, a regex for the duration (handle "45 min", "45min" and "1.5h"), and keyword rules for the action, the parts and the follow-up. Score it on `val.jsonl` with `grade`.

You are done when you have a per-field accuracy table for the baseline. These are the numbers the fine-tuned model has to beat in the next section; the solution discusses why they are suspiciously high.

In [ ]:
# The generator: synthetic technician notes and their JSON records, so the training cells run without
# any API. The distillation exercise at the end of the lesson replaces it with distilled or hand-labelled data.
import json, random

random.seed(0)
SYSTEM = ("Convert the technician note to JSON with keys "
          "instrument_id, action, parts, duration_min, follow_up.")
JOBS = [
    ("maintenance", ["desiccant"], ["swapped desiccant", "new desiccant cartridge", "desiccant replaced"]),
    ("maintenance", ["filter wheel belt"], ["new belt on filter wheel", "replaced FW belt", "fw belt swapped"]),
    ("maintenance", ["desiccant", "filter wheel belt"], ["swapped desiccant + new belt on filter wheel",
                                                         "desiccant and fw belt both replaced"]),
    ("maintenance", ["entrance window"], ["cleaned entrance window", "window cleaned, lots of dust"]),
    ("repair", ["gps antenna"], ["replaced gps antenna", "new GPS antenna fitted"]),
    ("repair", ["cooler fan"], ["cooler fan dead, replaced", "swapped cooler fan"]),
    ("repair", ["modem"], ["swapped modem", "modem replaced (no signal)"]),
    ("firmware_update", [], ["firmware 2.3->2.4", "fw update to 2.4", "updated firmware to v2.4"]),
    ("inspection", [], ["visual check, nothing found", "routine inspection", "looked over, all fine"]),
]
FOLLOW = [(None, ["all ok", "", "done"]),
          ("recheck in 2 weeks", ["check again in 2 wks", "recheck in two weeks"]),
          ("return with spare belt", ["need to come back w/ spare belt", "return with spare belt"]),
          ("monitor data upload", ["keep an eye on uploads", "monitor upload tonight"])]
SITES = ["coastal", "urban", "desert", "alpine", ""]

def make_example():
    action, parts, phrasings = random.choice(JOBS)
    follow, follow_phrasings = random.choice(FOLLOW)
    minutes = random.choice([10, 15, 20, 30, 45, 60, 90, None])
    if minutes is None:
        dur = ""
    elif minutes >= 60 and random.random() < 0.5:
        dur = f"{minutes / 60:g}h"
    else:
        dur = random.choice([f"{minutes} min", f"{minutes}min", f"{minutes} mins"])
    iid = f"P-{random.randint(100, 299)}"
    pieces = [random.choice(phrasings), dur, random.choice(follow_phrasings)]
    pieces = [p for p in pieces if p]
    head = f"{iid} {random.choice(SITES)}".strip()
    note = f"{head}{random.choice(['. ', ': ', ' - '])}" + ", ".join(pieces)
    if random.random() < 0.3:
        note = note.lower()
    target = {"instrument_id": iid, "action": action, "parts": parts,
              "duration_min": minutes, "follow_up": follow}
    return {"messages": [{"role": "system", "content": SYSTEM},
                         {"role": "user", "content": note},
                         {"role": "assistant", "content": json.dumps(target)}]}

for name, n in [("train.jsonl", 400), ("val.jsonl", 60)]:
    with open(name, "w") as f:
        for _ in range(n):
            f.write(json.dumps(make_example()) + "\n")
print(open("train.jsonl").readline()[:300])

#### Solution: a solution

In [ ]:
# Step 1: read ten examples.
for line in open("train.jsonl").readlines()[:10]:
    ex = json.loads(line)
    print(ex["messages"][1]["content"])
    print("   ->", ex["messages"][2]["content"])

The generator varies the instrument id, the site, the separator, the job phrasing, the duration format and the follow-up, and lower-cases some notes. It never produces typos, unknown parts, two instruments in one note, or a follow-up in free text.

In [ ]:
import json, re

FIELDS = ["instrument_id", "action", "parts", "duration_min", "follow_up"]

def grade(pred: str, gold: dict) -> dict:
    try:
        p = json.loads(pred)
    except json.JSONDecodeError:
        return {"valid": 0, **{f: 0 for f in FIELDS}}
    return {"valid": 1, **{f: int(p.get(f) == gold.get(f)) for f in FIELDS}}

PARTS = [("desiccant", "desiccant"), ("belt", "filter wheel belt"), ("window", "entrance window"),
         ("gps", "gps antenna"), ("fan", "cooler fan"), ("modem", "modem")]
FOLLOW = [("wk", "recheck in 2 weeks"), ("two weeks", "recheck in 2 weeks"),
          ("spare belt", "return with spare belt"), ("upload", "monitor data upload")]

def rule_convert(note: str) -> str:
    t = note.lower()
    m = re.search(r"p-\d{3}", t)
    dur = re.search(r"(\d+(?:\.\d+)?)\s*(h|mins?)\b", t)
    minutes = None if not dur else (float(dur[1]) * 60 if dur[2] == "h" else float(dur[1]))
    parts = [name for key, name in PARTS if key in t]
    if "firmware" in t or ("fw" in t and "belt" not in t):
        action = "firmware_update"
    elif any(w in t for w in ["inspection", "visual check", "looked over"]):
        action = "inspection"
    elif any(p in ("gps antenna", "cooler fan", "modem") for p in parts):
        action = "repair"
    else:
        action = "maintenance"
    follow = next((v for k, v in FOLLOW if k in t), None)
    return json.dumps({"instrument_id": m[0].upper() if m else None, "action": action, "parts": parts,
                       "duration_min": None if minutes is None else int(minutes), "follow_up": follow})

def evaluate(convert_fn, path="val.jsonl"):
    rows = [json.loads(l) for l in open(path)]
    totals = {"valid": 0, **{f: 0 for f in FIELDS}}
    for ex in rows:
        note, gold = ex["messages"][1]["content"], json.loads(ex["messages"][2]["content"])
        for k, v in grade(convert_fn(note), gold).items():
            totals[k] += v
    return {k: round(v / len(rows), 3) for k, v in totals.items()}

print("rule baseline:", evaluate(rule_convert))

Before you trust the table, read the notes behind it.

In [ ]:
# Read the misses: the table says which field, the notes say why.
val_rows = [json.loads(l) for l in open("val.jsonl")]
misses = []
for ex in val_rows:
    note, gold = ex["messages"][1]["content"], json.loads(ex["messages"][2]["content"])
    wrong = [f for f, ok in grade(rule_convert(note), gold).items() if not ok]
    if wrong:
        misses.append((wrong, note))
print(len(misses), "of", len(val_rows), "notes have a wrong field")
for wrong, note in misses[:5]:
    print(" ", wrong, note)

def rule_convert_v2(note: str) -> str:
    """The same rules, except that a belt named in the follow-up is not a part that was fitted."""
    out = json.loads(rule_convert(note))
    body = note.lower().replace("spare belt", "")
    out["parts"] = [name for key, name in PARTS if key in body]
    if "fw" in body and "belt" not in body:
        out["action"] = "firmware_update"
    return json.dumps(out)

print("rule baseline v2:", evaluate(rule_convert_v2))

The lesson's rules are perfect on validity, instrument id, duration and follow-up, and miss about a quarter of the parts: the follow-up "return with spare belt" names a belt that was never fitted, and once, in a firmware note, it also changes the action. One more rule fixes both, and the v2 baseline scores 100% on every field.

That is the lesson rather than the result: the generator has nine kinds of job and four kinds of follow-up, so a short list of keywords covers it. Real notes have typos ("dessicant"), new parts, two instruments in one line and free-text follow-ups that no keyword list anticipates. A fine-tuned model earns its place on *that* variation, which is why the distillation exercise at the end of this lesson insists on 30 hand-written notes as the eval set. Whatever you fine-tune, keep `evaluate`: the same function scores the base model, the fine-tuned model and the large model.

### Training with Hugging Face TRL and PEFT

Run this on a GPU (a free Colab T4 is enough for a 0.5B model). Library APIs evolve quickly; if an argument name has changed, check the TRL documentation for your installed version.

Run in a terminal:

```bash
pip install -U transformers datasets peft trl accelerate bitsandbytes
```

In [ ]:
# Needs a GPU runtime (Colab: Runtime > Change runtime type > GPU)
import torch
from datasets import load_dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

model_id = "Qwen/Qwen2.5-0.5B-Instruct"         # small instruction-tuned open model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map="auto")

data = load_dataset("json", data_files={"train": "train.jsonl", "validation": "val.jsonl"})

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules="all-linear",                 # LoRA on every linear layer
    task_type="CAUSAL_LM",
)
args = SFTConfig(
    output_dir="notes-to-json-lora",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,               # effective batch of 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,                                   # use bf16=True instead on Ampere or newer GPUs
)
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=data["train"],
    eval_dataset=data["validation"],
    peft_config=lora,
    processing_class=tokenizer,
)
trainer.train()
trainer.save_model("notes-to-json-lora")         # saves only the adapter: a few MB

The trainer applies the model's chat template to each conversation. By default it computes the loss on the whole sequence, including the system prompt and the note; for instruction data you usually want the loss only on the assistant's reply. TRL's `SFTConfig` has an `assistant_only_loss` option for this (check the documentation for your version), and lesson 19 builds the loss mask by hand.

For a larger model on limited memory, load it in 4-bit (QLoRA) instead:

In [ ]:
# Needs a GPU runtime (Colab: Runtime > Change runtime type > GPU)
from transformers import BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-7B-Instruct",
                                             quantization_config=bnb, device_map="auto")
# then the same LoraConfig / SFTConfig / SFTTrainer, typically with gradient_checkpointing=True in SFTConfig

### Watch the curves

Training loss should fall steadily. Validation loss should fall and then flatten; if it starts rising while training loss keeps falling, you are overfitting (lesson 03): fewer epochs, more data, or lower rank. A training loss near zero on a few hundred examples is a warning sign, not a triumph.

### Use and evaluate the result

In [ ]:
# Needs a GPU runtime (Colab: Runtime > Change runtime type > GPU)
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map="auto")
tuned = PeftModel.from_pretrained(base, "notes-to-json-lora")

def convert(note: str, m) -> str:
    msgs = [{"role": "system", "content": "Convert the technician note to JSON with keys "
                                          "instrument_id, action, parts, duration_min, follow_up."},
            {"role": "user", "content": note}]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(m.device)
    out = m.generate(ids, max_new_tokens=200, do_sample=False)
    return tokenizer.decode(out[0, ids.shape[1]:], skip_special_tokens=True)

print(convert("P-204: firmware 2.3->2.4, rebooted, all ok. 15min", tuned))
merged = tuned.merge_and_unload()                # fold LoRA into the weights for deployment
merged.save_pretrained("notes-to-json-merged")

Then run your lesson 17 harness on the held-out eval set, comparing three systems: the base small model with your best prompt, the fine-tuned small model, and the large model you distilled from. The question is never "did the loss go down" but "is the fine-tuned small model good enough on the eval set, at its cost and latency".

## Beyond supervised fine-tuning

SFT teaches by imitation. Two families of technique start where imitation stops. This section is a map; lesson 19 is the territory.

### Preference tuning

Sometimes it is easier to say which of two outputs is better than to write the perfect one. **Preference tuning** trains on triples of a prompt, a **chosen** response and a **rejected** one. The practical workhorse is direct preference optimisation (DPO), usually run after SFT; TRL's `DPOTrainer` takes the same model, tokeniser and LoRA configuration as the SFT code above. Lesson 19 derives the DPO loss, implements it from scratch, and goes on to reward models and GRPO.

### Distillation

**Knowledge distillation** trains a small "student" to imitate a large "teacher". The simplest version is what this lesson does: train on the teacher's outputs. The classic version also matches the teacher's full probability distribution over tokens with a KL-divergence loss, which needs the teacher's logits.

### Hosted fine-tuning

Several API providers fine-tune their own models: upload a JSONL file in their chat format, launch a job, get a model id. You give up control of the method and weights in exchange for zero infrastructure. The same rules apply: clean data, a held-out eval, and a comparison against a well-prompted baseline.

### Serving adapters

Because adapters are small, one base model can serve many of them. Servers such as vLLM load several LoRA adapters and pick one per request (lesson 21). A merged model can also be quantised for CPU or edge serving (lesson 20). Re-run your eval set after quantising, exactly as after fine-tuning.

### Things that go wrong

- **Catastrophic forgetting**: heavy fine-tuning on a narrow task degrades general abilities. LoRA, low learning rates and few epochs reduce it.
- **Format mismatch**: training with one chat template and serving with another silently ruins results. Always use `apply_chat_template` from the same tokeniser.
- **Data leakage** between training and eval examples gives flattering scores.
- **Safety regressions**: fine-tuning can weaken refusal behaviour, even on benign data. Include safety cases in your eval set (lesson 22).

### Hands-on: distil a large model into a small one

1. Write 30 realistic technician notes by hand (vary style, typos, abbreviations, and include some notes with missing information).
2. Use a large model via API to generate 400 more diverse notes and their JSON conversions. Review 40 at random and fix or drop bad ones.
3. Fine-tune `Qwen/Qwen2.5-0.5B-Instruct` with LoRA as above.
4. On your 30 hand-written notes (the eval set, never trained on), measure: JSON validity, exact match per field, and latency for the base small model with a few-shot prompt, the fine-tuned small model, and the large model.
5. Write one paragraph recommending which to deploy, and why.

#### Solution: what to expect and how to grade it

In [ ]:
# Needs a GPU runtime (Colab: Runtime > Change runtime type > GPU)
# grade() and evaluate() come from the baseline exercise; convert() from "Use and evaluate the result"
fresh_base = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map="auto")
print("base model :", evaluate(lambda note: convert(note, fresh_base)))
print("fine-tuned :", evaluate(lambda note: convert(note, merged)))

Typical outcome: the base 0.5B model with a few-shot prompt produces valid JSON most of the time but gets fields wrong or invents values; the fine-tuned model is close to 100% valid and approaches the large model on `instrument_id`, `duration_min` and `parts`, while still trailing on free-text fields like `follow_up`. It runs on a laptop CPU at a fraction of the cost. A sound recommendation might be: deploy the fine-tuned small model, route notes it fails to parse (or flags as low confidence) to the large model, and keep adding corrected failures to the training set for the next round. The exact numbers matter less than having measured them on data the model never saw.

## Check yourself

Take the quiz at the end of [the lesson page](../../lessons/18-fine-tuning.html). Then close everything and write three or four sentences in your learning log about what you learned, from memory.